In [0]:
import pandas as pd
import xgboost as xgb
import mlflow
import mlflow.xgboost
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import OrdinalEncoder
from sklearn.utils.class_weight import compute_sample_weight

print("1. Loading Gold Master Dataset...")
gold_df = spark.table("aviation_project.gold_master_dataset").toPandas()

# Sort chronologically to prevent time-series data leakage
gold_df = gold_df.sort_values("scheduled_time_utc").reset_index(drop=True)

In [0]:
gold_df.head()

In [0]:
gold_df["flight_status"].value_counts()

In [0]:
print("2. Defining Features and Target...")
target = 'flight_status'
features = [
    'distance_miles', 'origin_temp', 'origin_wind', 'origin_precip',
    'dest_temp', 'dest_wind', 'dest_precip', 'time_sin', 'time_cos',
    'is_holiday', 'month', 'day_of_month', 'day_of_week',
    'airline_code', 'origin_condition', 'dest_condition'
]

X = gold_df[features].copy()
y = gold_df[target].copy()

# Merge Class 3 (Diverted) into Class 2 (Canceled)
print("   -> Merging 'Diverted' into 'Canceled' due to low sample volume...")
y = y.replace(3, 2)
target_names = ["On Time", "Delayed", "Canceled/Diverted"]

print("   -> Encoding categorical variables to integers...")
categorical_cols = ['airline_code', 'origin_condition', 'dest_condition']
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X[categorical_cols] = encoder.fit_transform(X[categorical_cols])

# Save the encoder for Script 11 (Predicting your flight)
joblib.dump(encoder, '/Volumes/workspace/aviation_project/raw_data/ordinal_encoder.pkl')

In [0]:
print("3. Executing Chronological Train/Test Split (80/20)...")
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"   -> Train: {len(X_train)} rows | Test: {len(X_test)} rows")

In [0]:
# ---------------------------------------------------------
# Part 4: The Sample Weight Fix & Model Training
# ---------------------------------------------------------
print("4. Calculating Class Penalties (Sample Weights)...")
# This automatically balances the weights so rare events are heavily penalized if missed
train_sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

print("5. Initializing MLflow and Training XGBoost Model...")
xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    early_stopping_rounds=20,
    n_estimators=200,
    learning_rate=0.05,
    max_depth=8,
    random_state=42
)

In [0]:
with mlflow.start_run(run_name="LIT_Flight_Delay_XGBoost_Weighted"):
    
    xgb_model.fit(
        X_train, y_train,
        sample_weight=train_sample_weights, # INJECTING THE PENALTIES HERE
        eval_set=[(X_test, y_test)],
        verbose=False
    )
    
    print("6. Evaluating Model Performance...")
    predictions = xgb_model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    print(f"   -> Model Test Accuracy: {accuracy:.4f}")
    
    from mlflow.models.signature import infer_signature
    train_predictions = xgb_model.predict(X_train)
    signature = infer_signature(X_train, train_predictions)
    
    mlflow.log_param("max_depth", xgb_model.max_depth)
    mlflow.log_param("learning_rate", xgb_model.learning_rate)
    mlflow.log_param("class_weights", "balanced")
    mlflow.log_metric("accuracy", accuracy)
    
    mlflow.xgboost.log_model(
        xgb_model=xgb_model,
        artifact_path="xgboost-model",
        signature=signature
    )

In [0]:
print("\n6. SUCCESS! Model trained and logged to MLflow.")
print("\nClassification Report (Test Data):")
print(classification_report(y_test, predictions, target_names=["On Time", "Delayed", "Canceled"]))

In [0]:
print("\n8. Generating Confusion Matrix...")
cm = confusion_matrix(y_test, predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)

# Plotting with a clean blue color map
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(cmap='Blues', values_format='d', ax=ax)
plt.title('Flight Prediction Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Status', fontsize=12)
plt.ylabel('True Status', fontsize=12)
plt.tight_layout()
plt.show()

In [0]:
# ---------------------------------------------------------
# Part 8: Normalized Confusion Matrix Visualization
# ---------------------------------------------------------
print("\n8. Generating Normalized Confusion Matrix...")
# normalize='true' calculates the ratio across the true status (rows sum to 1.0)
cm = confusion_matrix(y_test, predictions, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)

# Plotting with a clean blue color map
fig, ax = plt.subplots(figsize=(8, 6))
# Changed values_format to '.2f' to beautifully display the float values (e.g., 0.43)
disp.plot(cmap='Blues', values_format='.2f', ax=ax)
plt.title('Normalized Flight Prediction Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Status', fontsize=12)
plt.ylabel('True Status', fontsize=12)
plt.tight_layout()
plt.show()